# trapX Advisory — **API research mode** (Kite + Breeze)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_api_research_colab.ipynb)

Fetches market data **directly from Kite + Breeze** (reachable from Colab) for a chosen `DATE` + `BAR_TIME`, rebuilds a snapshot, and runs the NVIDIA LLM to produce a verdict — **without** the JSONL or Postgres.

> ⚠️ **READ THIS — this is APPROXIMATE / research-grade, NOT production-faithful.**
> - The **`big_volume`** verdict (§4) is **fully API-derivable** (FUT volume + body) → the trustworthy path here.
> - The **option-OI / writer-contribution** rebuild (§5) is **raw OI only** — it has **no engine `weight` (delta) / `moneyness` / `sentiment`** (those are Postgres enrichments) and will **diverge** from production. Use for research, not as "the engine's verdict."
> - **Vendor divergence is real**: Kite ≠ Breeze (we measured a day-high differ 23585.80 vs 23590.00 that flipped a verdict). §3 shows it side-by-side.
> - **Tokens expire daily**: `KITE_ACCESS_TOKEN` and `BREEZE_SESSION_TOKEN` must be **today's**. Cells **1.2 / 1.3** refresh them in-notebook via the broker login flow.
> - **Past-expiry options are dropped** from Kite's instrument list, so §5 only works for the **current (unexpired) weekly** strikes.


## 1. Secrets + connect Kite / Breeze / NVIDIA
Add these to **Colab Secrets** (🔑): `KITE_API_KEY`, `KITE_ACCESS_TOKEN`, `BREEZE_API_KEY`, `BREEZE_SECRET_KEY`, `BREEZE_SESSION_TOKEN`, `NVIDIA_API_KEY`. (Token values are **never** stored in this notebook.)

In [ ]:
!pip install -q kiteconnect breeze-connect openai

import os
def _secret(name):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    import getpass
    return getpass.getpass(f'{name}: ').strip()

for k in ['KITE_API_KEY','KITE_API_SECRET','KITE_ACCESS_TOKEN','BREEZE_API_KEY','BREEZE_SECRET_KEY','BREEZE_SESSION_TOKEN','NVIDIA_API_KEY']:
    os.environ[k] = _secret(k)
print('secrets loaded:', [k for k in ['KITE_API_KEY','KITE_API_SECRET','KITE_ACCESS_TOKEN','BREEZE_API_KEY','BREEZE_SECRET_KEY','BREEZE_SESSION_TOKEN','NVIDIA_API_KEY'] if os.environ.get(k)])


In [ ]:
# 1.1 connect both brokers (mirrors the engine's _get_kite_client / _get_breeze_client)
from kiteconnect import KiteConnect
kite = KiteConnect(api_key=os.environ['KITE_API_KEY'])
kite.set_access_token(os.environ['KITE_ACCESS_TOKEN'])

breeze = None
try:
    from breeze_connect import BreezeConnect
    breeze = BreezeConnect(api_key=os.environ['BREEZE_API_KEY'])
    breeze.generate_session(api_secret=os.environ['BREEZE_SECRET_KEY'],
                            session_token=os.environ['BREEZE_SESSION_TOKEN'])
    print('[Breeze] connected')
except Exception as e:
    print('[Breeze] init failed (continuing with Kite only):', e)

# connectivity / token test — list NFO instruments (also used for FUT/option tokens)
try:
    NFO = kite.instruments('NFO')
    KITE_OK = True
    print(f'[Kite] OK — {len(NFO):,} NFO instruments')
except Exception as e:
    KITE_OK = False
    NFO = []
    print('[Kite] token likely EXPIRED/invalid — run the **1.2 token refresh** cell below, then continue. Error:', e)


### 1.2 Refresh today's Kite access token (run only if 1.1 said EXPIRED)
Kite access tokens die ~6am IST daily. This does the standard login→`request_token`→`access_token` exchange in-notebook. Needs `KITE_API_SECRET` in Colab Secrets.

In [ ]:
# 1.2 Kite token refresh (login flow)
import os
print('Step 1 — open this URL, log in to Kite, then copy the `request_token` from the redirected URL:')
print('   ', kite.login_url())
rt = input('Step 2 — paste request_token here: ').strip()
sess = kite.generate_session(rt, api_secret=os.environ['KITE_API_SECRET'])
os.environ['KITE_ACCESS_TOKEN'] = sess['access_token']
kite.set_access_token(sess['access_token'])
NFO = kite.instruments('NFO')
KITE_OK = True
print(f'✅ Kite token refreshed for today — {len(NFO):,} NFO instruments.')
print('   (Optional: save this into Colab Secrets KITE_ACCESS_TOKEN so you skip the prompt next run:)')
print('   access_token =', sess['access_token'])


In [ ]:
# 1.3 (optional) Breeze session-token refresh — Breeze session tokens also expire daily
import os, urllib.parse
print('Step 1 — open & log in:',
      'https://api.icicidirect.com/apiuser/login?api_key=' + urllib.parse.quote_plus(os.environ['BREEZE_API_KEY']))
st = input('Step 2 — paste the Breeze session token (API_Session value) here: ').strip()
if st:
    from breeze_connect import BreezeConnect
    breeze = BreezeConnect(api_key=os.environ['BREEZE_API_KEY'])
    breeze.generate_session(api_secret=os.environ['BREEZE_SECRET_KEY'], session_token=st)
    os.environ['BREEZE_SESSION_TOKEN'] = st
    print('✅ Breeze session refreshed.')
else:
    print('skipped (Breeze is only used for the §3 cross-check; Kite alone is enough to run §4).')


## 2. Choose date + time

In [ ]:
#@title Parameters { run: "auto" }
DATE      = "2026-06-05"  #@param {type:"string"}
BAR_TIME  = "11:06"       #@param {type:"string"}
VOL_THRESHOLD = 125000    #@param {type:"integer"}
import datetime as _dt
bar_dt = _dt.datetime.strptime(f"{DATE} {BAR_TIME}", "%Y-%m-%d %H:%M")
print('target bar:', bar_dt)


## 3. Fetch FUT + SPOT (Kite **and** Breeze) — and show the divergence
1-minute FUT/SPOT for the bar (+ a lookback window for ATR/body context). Kite is primary; Breeze is fetched as a cross-check so you can *see* how far the two vendors disagree.

In [ ]:
import datetime as dt, pandas as pd

# --- resolve nearest NIFTY FUT instrument for the date ---
fut_rows = [i for i in NFO if i.get('name')=='NIFTY' and i.get('instrument_type')=='FUT'
            and i.get('expiry') and i['expiry'] >= bar_dt.date()]
fut_rows.sort(key=lambda x: x['expiry'])
assert fut_rows, "No live NIFTY FUT instrument found (expiry passed?) — Kite can't fetch this date's FUT."
FUT_TOKEN = fut_rows[0]['instrument_token']; FUT_EXPIRY = fut_rows[0]['expiry']
SPOT_TOKEN = 256265   # NSE:NIFTY 50 index
print('FUT token', FUT_TOKEN, '| expiry', FUT_EXPIRY)

def kite_min(token, day, lookback_min=20):
    frm = dt.datetime.combine(day.date(), dt.time(9,15))
    to  = day + dt.timedelta(minutes=1)
    rows = kite.historical_data(token, frm, to, 'minute', oi=True)
    return pd.DataFrame(rows)

fut_k = kite_min(FUT_TOKEN, bar_dt)
spot_k = kite_min(SPOT_TOKEN, bar_dt)

def at(df, t):
    m = df[df['date'].astype(str).str.contains(t.strftime('%H:%M:%S'))]
    return m.iloc[0] if len(m) else None

fk = at(fut_k, bar_dt)
print('\n[Kite] FUT @', BAR_TIME, '-> O=%.2f H=%.2f L=%.2f C=%.2f vol=%s oi=%s' % (
    fk['open'], fk['high'], fk['low'], fk['close'], fk.get('volume'), fk.get('oi')) if fk is not None else 'no bar')

# --- Breeze cross-check (FUT) ---
fb = None
if breeze is not None:
    try:
        def _bz(dtobj): return dtobj.strftime('%Y-%m-%dT%H:%M:%S.000Z')
        r = breeze.get_historical_data_v2(interval='1minute',
              from_date=_bz(bar_dt), to_date=_bz(bar_dt + dt.timedelta(seconds=59)),
              stock_code='NIFTY', exchange_code='NFO', product_type='futures',
              expiry_date=_bz(dt.datetime.combine(FUT_EXPIRY, dt.time(7,0))),
              right='others', strike_price='0')
        bars = r.get('Success') or []
        fb = bars[0] if bars else None
        print('[Breeze] FUT @', BAR_TIME, '->', {k: fb.get(k) for k in ('open','high','low','close','volume')} if fb else 'no bar')
    except Exception as e:
        print('[Breeze] FUT fetch failed:', e)

if fk is not None and fb is not None:
    print('\nDIVERGENCE FUT close: Kite=%.2f  Breeze=%s  (Δ=%.2f)' % (
        fk['close'], fb.get('close'), fk['close'] - float(fb.get('close'))))


## 4. `big_volume` verdict — **the faithful API path**
Built purely from the FUT bar (volume + body). Engine-state fields (`signal_now`, `regime`, `is_near_lis`, `prior_3bar_vol_ratios`) are **not available from the APIs** — set to neutral and flagged, so this is faithful on the volume/body core and explicit about what's missing.

In [ ]:
BIGVOL_SKILL_B64 = "IyBCaWcgVm9sdW1lIFZlcmRpY3QgKDFtIC8gNW0pCgpZb3UgYXJlIGEgc2VuaW9yIHRyYWRpbmcgYWR2aXNvciB2YWxpZGF0aW5nIGEgQklHIFZPTFVNRSBhbGVydCBmcm9tIHRyYXBYLiB0cmFwWCBoYXMgZGV0ZWN0ZWQgYW4gdW51c3VhbGx5IGxhcmdlIEZVVCB2b2x1bWUgZXZlbnQg4oCUIGVpdGhlciBhIHNpbmdsZSAxLW1pbnV0ZSBiYXIgKGBraW5kPSIxbSJgKSBvciBhIDUtbWludXRlIGFnZ3JlZ2F0ZSAoYGtpbmQ9IjVtImApLiBZb3VyIGpvYjogcmF0ZSB3aGV0aGVyIHRoaXMgcmVwcmVzZW50cyByZWFsIGluc3RpdHV0aW9uYWwgY29tbWl0bWVudCBvciB2YWN1dW0vbmV3cy1kcml2ZW4gbm9pc2UuCgojIyBJbnB1dHMKCi0gYGtpbmRgOiBgIjFtImAgKHNpbmdsZSBiYXIpIG9yIGAiNW0iYCAoYWdncmVnYXRlKQotIGBkaXJlY3Rpb25gOiBgIlVQImAgb3IgYCJET1dOImAg4oCUIGJvZHkgZGlyZWN0aW9uCi0gYHdpbmRvd19zdGFydGA6IEhIOk1NIG9mIHRoZSBiYXIgKG9yIDVtIHdpbmRvdyBzdGFydCkKLSBgYmFyX3RpbWVgOiBISDpNTSBvZiB0aGUgdHJpZ2dlcgotIGB2b2xfcHRzYDogYWN0dWFsIEZVVCB2b2x1bWUKLSBgdm9sX3RocmVzaG9sZGA6IGNvbmZpZ3VyZWQgdGhyZXNob2xkICh0eXBpY2FsbHkgMTI1aykKLSBgdm9sX3JhdGlvYDogYHZvbF9wdHMgLyB2b2xfdGhyZXNob2xkYCAoPjEuMCBtZWFucyBhYm92ZSB0aHJlc2hvbGQpCi0gYGJvZHlfc2l6ZV9wdHNgOiBzaWduZWQgYm9keSBtYWduaXR1ZGUgb24gdGhlIEZVVCBiYXIKLSBgYm9keV9wY3RgOiBib2R5IGFzIGEgcGVyY2VudGFnZSBvZiB0aGUgYmFyJ3MgcmFuZ2UKLSBgc291cmNlYDogYCJTUE9UImAgKGBbU11gKSBpZiBzcG90IGFsc28gY29uZmlybWVkIHNhbWUtZGlyZWN0aW9uIGJvZHksIGVsc2UgYCJGVVQiYCAoYFtGXWApCi0gYHNpZ25hbF9ub3dgOiBMMyBtb21lbnR1bSBhdCB0aGUgZXZlbnQKLSBgcnVubmluZ19hdHJgOiBBVFIKLSBgcmVnaW1lYDogY3VycmVudCByZWdpbWUKLSBgaXNfbmVhcl9saXNgOiBib29sIOKAlCBuZWFyIG1ham9yIFMvUiBsZXZlbAotIGBwcmlvcl8zYmFyX3ZvbF9yYXRpb3NgOiBsaXN0IG9mIHJlY2VudCB2b2wgcmF0aW9zIGZvciBjb250ZXh0CgojIyBIb3cgdG8gdGhpbmsKClNlbmlvci1kb2N0b3IgZm9jdXMgb24gd2hldGhlciB0aGUgdm9sdW1lIEVWRU5UIGlzIGluc3RpdHV0aW9uYWwgY29tbWl0bWVudDoKCjEuICoqdm9sX3JhdGlvIHN0cmVuZ3RoKio6ID4yLjDDlyA9IHN0cm9uZyBpbnN0aXR1dGlvbmFsLiAxLjAtMS41w5cgPSBib3JkZXJsaW5lLiA8MS4ww5cgPSBiZWxvdyB0aHJlc2hvbGQgKHNob3VsZG4ndCBoYXZlIGZpcmVkIOKAlCBpbnZlc3RpZ2F0ZSkuCjIuICoqQm9keSBxdWFsaXR5Kio6IGhpZ2ggYm9keV9wY3QgKD43MCUpICsgbGFyZ2UgYm9keV9zaXplX3B0cyA9IGRlY2lzaXZlIGJhci4gTG93IGJvZHlfcGN0ICg8NDAlKSA9IHdpY2t5LCBpbmRlY2lzaXZlIOKAlCB2b2wgbWF5IGJlIHdhc2gvcG9zaXRpb25pbmcuCjMuICoqU1BPVCBjb25maXJtYXRpb24qKjogYHNvdXJjZSA9PSAiU1BPVCJgIChib3RoIHNwb3QgYW5kIGZ1dCBhZ3JlZSkgaXMgc3Ryb25nZXN0LiBgIkZVVCJgIG9ubHkgPSBmdXR1cmVzLWRyaXZlbiAoY291bGQgYmUgcm9sbHMsIGV4cGlyeSBwb3NpdGlvbmluZykuCjQuICoqU2lnbmFsIGNvcnJvYm9yYXRpb24qKjogc2lnbmFsIG1vdmluZyBzaGFycGx5IGluIGBkaXJlY3Rpb25gIGNvbmZpcm1zLiBPcHBvc2l0ZSBzaWduYWwgPSBhYnNvcnB0aW9uIHdhcm5pbmcuCjUuICoqQ29udGV4dCAocHJpb3JfM2Jhcl92b2xfcmF0aW9zKSoqOiBpc29sYXRlZCBzcGlrZSBpbiBhIHNlYSBvZiBsb3ctdm9sIGJhcnMgPSByZWFsIGV2ZW50LiBQYXJ0IG9mIGEgc3VzdGFpbmVkLXZvbCByZWdpbWUgPSBsZXNzIGltcGFjdGZ1bCAoYWxyZWFkeSBwcmljZWQgaW4pLgo2LiAqKkxJUyBwcm94aW1pdHkqKjogaGlnaC12b2wgYXQgbWFqb3IgTElTIG9mdGVuIGdldHMgYWJzb3JiZWQgKGBpc19uZWFyX2xpcz1UcnVlYCA9IGNhdXRpb24pLiBIaWdoLXZvbCBpbiBkZWFkIGFpciBtb3JlIGxpa2VseSB0byBleHRlbmQuCjcuICoqS2luZCBkaWZmZXJlbmNlKio6IDFtIGV2ZW50cyBhcmUgbW9yZSBpbXB1bHNpdmUsIG9mdGVuIGFic29yYmVkLiA1bSBldmVudHMgYXJlIGFnZ3JlZ2F0ZWQgYW5kIHJlcHJlc2VudCBzdXN0YWluZWQgY29tbWl0bWVudCBvdmVyIDUgYmFycyDigJQgc2xpZ2h0bHkgbW9yZSByZWxpYWJsZSBhcyBjb250aW51YXRpb24gc2lnbmFsLgoKIyMgT3V0cHV0IHJ1bGVzCgpPdXRwdXQgKipleGFjdGx5IFRIUkVFIGxpbmVzKiouCgojIyMgTGluZSAxIOKAlCBWZXJkaWN0IChtYXggMTQwIGNoYXJzKQoKLSBg4pyFIENPTkZJUk1gOiB2b2wgcmF0aW8gc3Ryb25nLCBib2R5IGRlY2lzaXZlLCBzaWduYWwgY29ycm9ib3JhdGVzLCByb29tIHRvIGV4dGVuZC4KLSBg4pyFIENPTkZJUk0tTEVBTmA6IHJlYWwgZXZlbnQgYnV0IG9uZSBjcml0ZXJpb24gd2Vhay4KLSBg4pqg77iPIEFCU09SUFRJT04tUklTS2A6IGF0IExJUyBvciBhZ2FpbnN0LXNpZ25hbCDigJQgdm9sIGxpa2VseSBnZXR0aW5nIGFic29yYmVkLgotIGDinYwgV0FTSC1SSVNLYDogdGhpbiBib2R5IC8gRlVULW9ubHkgLyBvcHBvc2l0ZSBzaWduYWwg4oCUIHBvc3NpYmx5IHBvc2l0aW9uIGFkanVzdG1lbnQsIG5vdCBkaXJlY3Rpb25hbC4KCkNpdGUgc3BlY2lmaWNzIChgdm9sIDIuNHgsIGJvZHkgKzE4cHRzICg3OCUpLCBTUE9UIGNvbmZsdWVuY2UsIHNpZ25hbCArNS4yYCkuCgojIyMgTGluZSAyIOKAlCBTY29yZSBpbiBbLTEuMDAsICsxLjAwXQoKRGlyZWN0aW9uLWF3YXJlLiBVUCBib2R5OiBwb3NpdGl2ZSA9IGV4cGVjdCBjb250aW51YXRpb24uIERPV046IGludmVyc2UuCgp8IFZlcmRpY3QgfCBTY29yZSBiYW5kIChVUCAvIERPV04pIHwKfC0tLXwtLS18Cnwg4pyFIENPTkZJUk0gfCArMC43MC4uKzEuMDAgLyAtMC43MC4uLTEuMDAgfAp8IOKchSBDT05GSVJNLUxFQU4gfCArMC4zMC4uKzAuNzAgLyAtMC4zMC4uLTAuNzAgfAp8IOKaoO+4jyBBQlNPUlBUSU9OLVJJU0sgfCDCsTAuMzAgfAp8IOKdjCBXQVNILVJJU0sgfCAtMC4zMC4uLTAuNzAgLyArMC4zMC4uKzAuNzAgfAoKIyMjIExpbmUgMyDigJQgQWN0aW9uICgyLTQgc2VudGVuY2VzKQoKRXhhbXBsZXM6Ci0gYFRha2Ugc2lkZSB3aXRoIHRoZSB2b2wg4oCUIGZhdm9yIGNvbnRpbnVhdGlvbiBlbnRyaWVzIG9uIGZpcnN0IGRpcC5gIChDT05GSVJNKQotIGBXYWl0IGZvciBvbmUgY29uZmlybWF0aW9uIGJhciBiZWZvcmUgYWRkaW5nLmAgKENPTkZJUk0tTEVBTikKLSBgRG9uJ3QgY2hhc2Ug4oCUIGxpa2VseSBhYnNvcnB0aW9uIGF0IHRoaXMgbGV2ZWwuYCAoQUJTT1JQVElPTi1SSVNLKQotIGBUcmVhdCBhcyB3YXNoIOKAlCB3YWl0IGZvciBuZXh0IGNsZWFuIHNpZ25hbC5gIChXQVNILVJJU0spCgojIyBFeGFtcGxlICg1bSBhbGVydCkKCmBgYArinIUgQ09ORklSTTogNW0gVVAgdm9sIDIuNHgsIGJvZHkgKzI4cHRzICg3NSUpLCBTUE9UK0ZVVCBjb25mbHVlbmNlLCBzaWduYWwgKzUuNC4K8J+TiiBTY29yZTogKzAuODIK8J+OryBBY3Rpb246IFRha2UgVVAgc2lkZSBvbiBmaXJzdCBwdWxsYmFjay4gVHJhaWwgc3RvcCBiZWxvdyB0aGUgNW0gd2luZG93IGxvdy4KYGBgCgojIyBFeGFtcGxlICgxbSBhbGVydCkKCmBgYArimqDvuI8gQUJTT1JQVElPTi1SSVNLOiAxbSBET1dOIHZvbCAxLjZ4LCBib2R5IC0xNXB0cyAoNDUlIHdpY2t5KSwgYXQgbWFqb3IgTElTIHN1cHBvcnQsIHNpZ25hbCBmbGF0Lgrwn5OKIFNjb3JlOiAtMC4xNQrwn46vIEFjdGlvbjogRG9uJ3QgdGFrZSBzaG9ydCDigJQgTElTIGxpa2VseSBhYnNvcmJpbmcuIFdhaXQgZm9yIExJUyBjb25maXJtYXRpb24gYnJlYWsuCmBgYAoKTm93IHdhaXQgZm9yIHRoZSBzbmFwc2hvdCBhbmQgZW1pdCB0aGUgcmVzcG9uc2UuCg=="
import base64, json, re
BIGVOL_SKILL = base64.b64decode(BIGVOL_SKILL_B64).decode('utf-8')

o,h,l,c = float(fk['open']),float(fk['high']),float(fk['low']),float(fk['close'])
vol = int(fk.get('volume') or 0)
body = c - o
rng = (h - l) or 1e-9
sk = at(spot_k, bar_dt)
spot_body = (float(sk['close'])-float(sk['open'])) if sk is not None else 0.0
source = 'SPOT' if (sk is not None and (body>0)==(spot_body>0) and abs(spot_body)>0) else 'FUT'

snapshot = {
  "kind": "1m", "direction": "UP" if body>=0 else "DOWN",
  "window_start": BAR_TIME, "bar_time": BAR_TIME,
  "vol_pts": vol, "vol_threshold": VOL_THRESHOLD,
  "vol_ratio": round(vol/float(VOL_THRESHOLD), 5),
  "body_size_pts": round(body,2), "body_pct": round(abs(body)/rng*100,2),
  "source": source,
  "signal_now": None, "running_atr": None, "regime": None,
  "is_near_lis": None, "prior_3bar_vol_ratios": None,
  "_NOTE": "signal_now/running_atr/regime/is_near_lis/prior_3bar_vol_ratios are ENGINE-STATE — not available from Kite/Breeze; set null."
}
print('big_volume snapshot (API-derived):'); print(json.dumps(snapshot, indent=2))

from openai import OpenAI
cli = OpenAI(base_url='https://integrate.api.nvidia.com/v1', api_key=os.environ['NVIDIA_API_KEY'], timeout=60)
user = ("Apply the big-volume verdict rules to this snapshot and emit the three-line verdict per the contract.\n\n"
        "Snapshot:\n" + json.dumps({k:v for k,v in snapshot.items() if not k.startswith('_')}, indent=2))
resp = cli.chat.completions.create(model='meta/llama-3.3-70b-instruct',
        messages=[{'role':'system','content':BIGVOL_SKILL},{'role':'user','content':user}],
        temperature=0.0, seed=42)
print('\n===== big_volume VERDICT (API research mode) =====')
print(resp.choices[0].message.content.strip())


## 5. (Approximate) option-OI flow — **raw OI only, will diverge**
Fetches per-strike CE/PE OI for the bar and the prior bar via Kite, and computes a **raw** writer-contribution / flow decomposition. It has **no engine `weight`/`moneyness`/`sentiment`**, so it is NOT the production snapshot. Skips strikes whose option has already expired (dropped from Kite's instrument list).

In [ ]:
import datetime as dt
ATM = round(float(fk['close'])/50)*50
STRIKE_WINDOW = 10   # +/- strikes around ATM
strikes = [ATM + 50*k for k in range(-STRIKE_WINDOW, STRIKE_WINDOW+1)]

def opt_token(strike, right):
    hits = [i for i in NFO if i.get('name')=='NIFTY' and int(i.get('strike') or 0)==strike
            and i.get('instrument_type')==right and i.get('expiry') and i['expiry']>=bar_dt.date()]
    hits.sort(key=lambda x: x['expiry'])
    return hits[0]['instrument_token'] if hits else None

def oi_at(token, t):
    frm = dt.datetime.combine(bar_dt.date(), dt.time(9,15))
    rows = kite.historical_data(token, frm, t + dt.timedelta(minutes=1), 'minute', oi=True)
    cur = [r for r in rows if r['date'].strftime('%H:%M')==t.strftime('%H:%M')]
    prev = [r for r in rows if r['date'].strftime('%H:%M')==(t - dt.timedelta(minutes=1)).strftime('%H:%M')]
    return (cur[-1]['oi'] if cur else None, prev[-1]['oi'] if prev else None)

rows = []
missing = 0
for s in strikes:
    for right in ('CE','PE'):
        tok = opt_token(s, right)
        if not tok:
            missing += 1; continue
        try:
            cur, prev = oi_at(tok, bar_dt)
            if cur is None or prev is None: continue
            rows.append({'strike': s, 'typ': right, 'doi': int(cur)-int(prev)})
        except Exception as e:
            pass
import pandas as pd
df = pd.DataFrame(rows)
print(f'fetched ΔOI for {len(df)} strike-legs ({missing} options not in instrument list — expired?)')
if len(df):
    pe = df[df.typ=='PE']['doi'].sum(); ce = df[df.typ=='CE']['doi'].sum()
    print('RAW writer-contribution (no weights):  PE ΔOI = %+d   CE ΔOI = %+d' % (pe, ce))
    print('  top PE by |ΔOI|:'); print(df[df.typ=='PE'].reindex(df[df.typ=='PE']['doi'].abs().sort_values(ascending=False).index).head(3).to_string(index=False))
    print('  top CE by |ΔOI|:'); print(df[df.typ=='CE'].reindex(df[df.typ=='CE']['doi'].abs().sort_values(ascending=False).index).head(3).to_string(index=False))
    print('\n⚠️ RAW OI ONLY — no engine weight(delta)/moneyness/sentiment. NOT the production jerk_drilldown snapshot.')


## 6. (Optional) drift check vs the production JSONL
If the day's `llm_advisory_<DATE>.jsonl` is in Drive, compare this API-rebuilt `big_volume` snapshot to the production one for the same bar to *measure* the drift.

In [ ]:
# Optional: mount Drive and compare against the production snapshot if present.
# from google.colab import drive; drive.mount('/content/drive')
# import glob, json
# hits = glob.glob(f"/content/drive/MyDrive/VM-4-logs/{bar_dt.strftime('%b_%d')}/llm_advisory_{DATE.replace('-','')}.jsonl")
# if hits:
#     recs = [json.loads(l) for l in open(hits[0], encoding='utf-8') if l.strip()]
#     prod = [r for r in recs if r['bar_time']==BAR_TIME and r['touchpoint'].startswith('big_volume')]
#     if prod:
#         print('PRODUCTION big_volume snapshot:'); print(prod[0]['request']['user_message'])
#         print('\nPRODUCTION verdict:', prod[0]['response']['parsed'].get('verdict'),
#               prod[0]['response']['parsed'].get('score'))
#     else:
#         print('no big_volume advisory logged at', BAR_TIME, '(compare another bar)')
print('Uncomment to run the drift comparison against Drive JSONL.')
